In [4]:
import pysus
import pandas as pd
import pyarrow as pa

print(f"pysus:   {pysus.__version__}")
print(f"pandas:  {pd.__version__}")
print(f"pyarrow: {pa.__version__}")

pysus:   2.1.0
pandas:  2.3.3
pyarrow: 24.0.0


In [10]:
import pysus
import inspect

print(inspect.signature(pysus.sih))
print()
print(inspect.getsource(pysus.sih))

(state: Literal['AC', 'AL', 'AP', 'AM', 'BA', 'CE', 'ES', 'GO', 'MA', 'MT', 'MS', 'MG', 'PA', 'PB', 'PR', 'PE', 'PI', 'RJ', 'RN', 'RS', 'RO', 'RR', 'SC', 'SP', 'SE', 'TO', 'DF'], year: int | list[int], month: int | list[int], group: str | None = None, **kwargs) -> pandas.core.frame.DataFrame

def sih(
    state: State,
    year: int | list[int],
    month: int | list[int],
    group: str | None = None,
    **kwargs,
) -> pd.DataFrame:
    """Fetch SIH hospital admissions for a state, year, month, and group."""
    return _fetch_data(
        dataset="sih",
        state=state.upper(),
        group=group,
        year=year,
        month=month,
    )



In [21]:
import ftplib
import os
from pathlib import Path

# Garante que a pasta existe
raw_path = Path('../data/raw')
raw_path.mkdir(parents=True, exist_ok=True)

# Arquivo: RD (reduzida) + SC + 23 (2023) + 01 (janeiro)
filename = 'RDSC2301.DBC'
local_file = raw_path / filename

print(f"Conectando ao FTP do DataSUS...")
ftp = ftplib.FTP('ftp.datasus.gov.br')
ftp.login()  # acesso anônimo — sem usuário/senha
ftp.cwd('/dissemin/publicos/SIHSUS/200801_/Dados/')

print(f"Baixando {filename}...")
with open(local_file, 'wb') as f:
    ftp.retrbinary(f'RETR {filename}', f.write)

ftp.quit()
print(f"Concluído! Arquivo salvo em: {local_file}")
print(f"Tamanho: {local_file.stat().st_size / 1024:.0f} KB")

Conectando ao FTP do DataSUS...
Baixando RDSC2301.DBC...
Concluído! Arquivo salvo em: ..\data\raw\RDSC2301.DBC
Tamanho: 3372 KB


In [28]:
from pysus.api.ftp.databases import SIH
print(inspect.getsource(SIH))

class SIH(Dataset):
    """Sistema de Informações Hospitalares (SIH)."""

    paths: list[Directory] = [
        Directory("/dissemin/publicos/SIHSUS/199201_200712/Dados"),
        Directory("/dissemin/publicos/SIHSUS/200801_/Dados"),
    ]
    group_definitions: dict[str, str] = {
        "RD": "AIH Reduzida",
        "RJ": "AIH Rejeitada",
        "SP": "Serviços Profissionais",
        "ER": "AIH Rejeitada com Erro",
    }

    @property
    def name(self) -> str:
        """Return the dataset short name."""
        return "SIH"

    @property
    def long_name(self) -> str:
        """Return the dataset full name in Portuguese."""
        return "Sistema de Informações Hospitalares"

    @property
    def description(self) -> str:
        """Return a description of the dataset's purpose."""
        return """
            O SIH processa as internações hospitalares financiadas pelo SUS.
        """

    def formatter(self, filename: str) -> dict[str, Any]:
        """Parse an SIH fil

In [31]:
import tempfile
import os
from pyreaddbc import dbc2dbf
from dbfread import DBF
import pandas as pd
from pathlib import Path

def decode_value(v):
    """Decodifica bytes para string usando cp1252 (padrão DataSUS)."""
    if isinstance(v, bytes):
        return v.decode('cp1252', errors='replace').replace('\x00', '').strip()
    if isinstance(v, str):
        return v.replace('\x00', '').strip()
    return v

dbc_file = Path('../data/raw/RDSC2301.DBC')

print("Convertendo DBC → DBF → DataFrame...")
with tempfile.TemporaryDirectory() as tmpdir:
    dbf_file = os.path.join(tmpdir, 'temp.dbf')
    dbc2dbf(str(dbc_file), dbf_file)
    dbf = DBF(dbf_file, encoding='cp1252', raw=True)
    df_amostra = pd.DataFrame(iter(dbf)).map(decode_value)

print(f"✓ Shape: {df_amostra.shape[0]:,} linhas × {df_amostra.shape[1]} colunas")
df_amostra.head(3)

Convertendo DBC → DBF → DataFrame...
✓ Shape: 44,570 linhas × 113 colunas


,UF_ZI,ANO_CMPT,MES_CMPT,ESPEC,CGC_HOSP,N_AIH,IDENT,CEP,MUNIC_RES,NASC,...,DIAGSEC9,TPDISEC1,TPDISEC2,TPDISEC3,TPDISEC4,TPDISEC5,TPDISEC6,TPDISEC7,TPDISEC8,TPDISEC9
0,420000,2023,01,01,84592369000988,4222102330303,1,89609000,421003,19860306,...,,0,0,0,0,0,0,0,0,0
1,420000,2023,01,01,84592369000988,4222102330325,1,89500785,420300,19781007,...,,0,0,0,0,0,0,0,0,0
2,420000,2023,01,01,84592369000988,4222102330424,1,89600000,420900,19670508,...,,0,0,0,0,0,0,0,0,0


In [32]:
print("=== SHAPE ===")
print(f"{df_amostra.shape[0]:,} linhas × {df_amostra.shape[1]} colunas")

print("\n=== TIPOS DE DADOS ===")
print(df_amostra.dtypes.value_counts())

print("\n=== NULOS POR COLUNA (top 15) ===")
nulls = df_amostra.isnull().sum()
print(nulls[nulls > 0].sort_values(ascending=False).head(15))

print("\n=== PRIMEIRAS COLUNAS ===")
print(df_amostra.columns.tolist()[:30])

=== SHAPE ===
44,570 linhas × 113 colunas

=== TIPOS DE DADOS ===
object    113
Name: count, dtype: int64

=== NULOS POR COLUNA (top 15) ===
Series([], dtype: int64)

=== PRIMEIRAS COLUNAS ===
['UF_ZI', 'ANO_CMPT', 'MES_CMPT', 'ESPEC', 'CGC_HOSP', 'N_AIH', 'IDENT', 'CEP', 'MUNIC_RES', 'NASC', 'SEXO', 'UTI_MES_IN', 'UTI_MES_AN', 'UTI_MES_AL', 'UTI_MES_TO', 'MARCA_UTI', 'UTI_INT_IN', 'UTI_INT_AN', 'UTI_INT_AL', 'UTI_INT_TO', 'DIAR_ACOM', 'QT_DIARIAS', 'PROC_SOLIC', 'PROC_REA', 'VAL_SH', 'VAL_SP', 'VAL_SADT', 'VAL_RN', 'VAL_ACOMP', 'VAL_ORTP']


In [33]:
# Todas as colunas
print("=== TODAS AS 113 COLUNAS ===")
for i, col in enumerate(df_amostra.columns):
    print(f"{i+1:3d}. {col}")

=== TODAS AS 113 COLUNAS ===
  1. UF_ZI
  2. ANO_CMPT
  3. MES_CMPT
  4. ESPEC
  5. CGC_HOSP
  6. N_AIH
  7. IDENT
  8. CEP
  9. MUNIC_RES
 10. NASC
 11. SEXO
 12. UTI_MES_IN
 13. UTI_MES_AN
 14. UTI_MES_AL
 15. UTI_MES_TO
 16. MARCA_UTI
 17. UTI_INT_IN
 18. UTI_INT_AN
 19. UTI_INT_AL
 20. UTI_INT_TO
 21. DIAR_ACOM
 22. QT_DIARIAS
 23. PROC_SOLIC
 24. PROC_REA
 25. VAL_SH
 26. VAL_SP
 27. VAL_SADT
 28. VAL_RN
 29. VAL_ACOMP
 30. VAL_ORTP
 31. VAL_SANGUE
 32. VAL_SADTSR
 33. VAL_TRANSP
 34. VAL_OBSANG
 35. VAL_PED1AC
 36. VAL_TOT
 37. VAL_UTI
 38. US_TOT
 39. DT_INTER
 40. DT_SAIDA
 41. DIAG_PRINC
 42. DIAG_SECUN
 43. COBRANCA
 44. NATUREZA
 45. NAT_JUR
 46. GESTAO
 47. RUBRICA
 48. IND_VDRL
 49. MUNIC_MOV
 50. COD_IDADE
 51. IDADE
 52. DIAS_PERM
 53. MORTE
 54. NACIONAL
 55. NUM_PROC
 56. CAR_INT
 57. TOT_PT_SP
 58. CPF_AUT
 59. HOMONIMO
 60. NUM_FILHOS
 61. INSTRU
 62. CID_NOTIF
 63. CONTRACEP1
 64. CONTRACEP2
 65. GESTRISCO
 66. INSC_PN
 67. SEQ_AIH5
 68. CBOR
 69. CNAER
 70. VINCPRE

In [34]:
colunas = df_amostra.columns.tolist()

# Palavras-chave de cada indicador
busca = {
    'TMI (dias internação)':    ['DIARIA', 'DT_INTER', 'DT_SAIDA'],
    'Mortalidade':              ['MORTE'],
    'Custo por AIH':            ['VAL'],
    'Estabelecimento':          ['CNES', 'CGC'],
    'Diagnóstico/especialidade':['DIAG', 'ESPEC', 'CID'],
    'Paciente':                 ['NASC', 'SEXO', 'IDADE'],
    'Readmissão':               ['AIH', 'IDENT'],
}

for indicador, termos in busca.items():
    encontradas = [c for c in colunas if any(t in c for t in termos)]
    print(f"\n{indicador}:")
    print(f"  {encontradas}")


TMI (dias internação):
  ['QT_DIARIAS', 'DT_INTER', 'DT_SAIDA']

Mortalidade:
  ['MORTE', 'CID_MORTE']

Custo por AIH:
  ['VAL_SH', 'VAL_SP', 'VAL_SADT', 'VAL_RN', 'VAL_ACOMP', 'VAL_ORTP', 'VAL_SANGUE', 'VAL_SADTSR', 'VAL_TRANSP', 'VAL_OBSANG', 'VAL_PED1AC', 'VAL_TOT', 'VAL_UTI', 'VAL_SH_FED', 'VAL_SP_FED', 'VAL_SH_GES', 'VAL_SP_GES', 'VAL_UCI']

Estabelecimento:
  ['CGC_HOSP', 'CNES']

Diagnóstico/especialidade:
  ['ESPEC', 'DIAG_PRINC', 'DIAG_SECUN', 'CID_NOTIF', 'CID_ASSO', 'CID_MORTE', 'DIAGSEC1', 'DIAGSEC2', 'DIAGSEC3', 'DIAGSEC4', 'DIAGSEC5', 'DIAGSEC6', 'DIAGSEC7', 'DIAGSEC8', 'DIAGSEC9']

Paciente:
  ['NASC', 'SEXO', 'COD_IDADE', 'IDADE']

Readmissão:
  ['N_AIH', 'IDENT', 'SEQ_AIH5']


In [35]:
colunas_chave = ['N_AIH', 'IDENT', 'DT_INTER', 'DT_SAIDA', 'QT_DIARIAS',
                 'MORTE', 'ESPEC', 'DIAG_PRINC', 'VAL_TOT', 'CNES', 'SEXO', 'IDADE']

for col in colunas_chave:
    unicos = df_amostra[col].nunique()
    amostra = df_amostra[col].value_counts().head(5).to_dict()
    print(f"\n{col} ({unicos} valores únicos):")
    for v, c in amostra.items():
        print(f"  '{v}' → {c:,} ocorrências")


N_AIH (44552 valores únicos):
  '4222101157043' → 3 ocorrências
  '4222101136187' → 3 ocorrências
  '4222105476072' → 3 ocorrências
  '4222101112878' → 3 ocorrências
  '4222105549618' → 2 ocorrências

IDENT (2 valores únicos):
  '1' → 44,417 ocorrências
  '5' → 153 ocorrências

DT_INTER (244 valores únicos):
  '20230116' → 986 ocorrências
  '20230109' → 965 ocorrências
  '20230110' → 938 ocorrências
  '20230111' → 927 ocorrências
  '20230104' → 906 ocorrências

DT_SAIDA (175 valores únicos):
  '20230113' → 1,016 ocorrências
  '20230111' → 984 ocorrências
  '20230112' → 953 ocorrências
  '20230119' → 949 ocorrências
  '20230106' → 944 ocorrências

QT_DIARIAS (69 valores únicos):
  '1' → 14,402 ocorrências
  '2' → 9,826 ocorrências
  '3' → 5,449 ocorrências
  '4' → 3,169 ocorrências
  '5' → 2,212 ocorrências

MORTE (2 valores únicos):
  '0' → 42,531 ocorrências
  '1' → 2,039 ocorrências

ESPEC (12 valores únicos):
  '01' → 18,566 ocorrências
  '03' → 15,766 ocorrências
  '02' → 5,951 oc